In [ ]:
"""
Enhanced Exploratory Data Analysis for Neuro-CXG
Comprehensive visualization of dataset quality, demographics, and processing pipeline
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from collections import Counter

# Add src to path
sys.path.append(str(Path.cwd().parents[0] / 'src'))
from config import (
    MASTER_MANIFEST, DATA_IMAGES, DATA_FINAL, 
    NODE_ATTRIBUTES_TEMPORAL, NODE_FEATURES_3D,
    CAUSAL_GRAPHS_DIR
)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = (15, 10)


def check_data_availability():
    """Check which datasets are available."""
    datasets = {
        'Master Manifest': MASTER_MANIFEST,
        'Raw Images': DATA_IMAGES,
        'Temporal Features': NODE_ATTRIBUTES_TEMPORAL,
        '3D Features': NODE_FEATURES_3D,
        'Causal Graphs': CAUSAL_GRAPHS_DIR
    }
    
    print("="*60)
    print("DATA AVAILABILITY CHECK")
    print("="*60)
    
    available = {}
    for name, path in datasets.items():
        exists = path.exists()
        status = "✓" if exists else "✗"
        print(f"{status} {name}: {path}")
        available[name] = exists
        
        if exists and path.is_dir():
            count = len(list(path.glob("*")))
            print(f"  └─ Contains {count} items")
    
    print("="*60 + "\n")
    return available


def analyze_demographics(manifest_df):
    """Analyze demographic distributions."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Demographic Analysis', fontsize=16, fontweight='bold')
    
    # 1. Diagnosis Distribution
    ax = axes[0, 0]
    dx_counts = manifest_df['DX_GROUP'].value_counts()
    colors = ['#FF6B6B', '#4ECDC4']
    ax.bar(['ASD', 'Control'], [dx_counts.get(1, 0), dx_counts.get(2, 0)], color=colors)
    ax.set_title('Diagnosis Distribution')
    ax.set_ylabel('Number of Subjects')
    for i, (label, count) in enumerate(zip(['ASD', 'Control'], [dx_counts.get(1, 0), dx_counts.get(2, 0)])):
        ax.text(i, count, str(count), ha='center', va='bottom', fontweight='bold')
    
    # 2. Age Distribution by Diagnosis
    ax = axes[0, 1]
    if 'AGE_AT_SCAN' in manifest_df.columns:
        valid_ages = manifest_df[manifest_df['AGE_AT_SCAN'] > 0]
        for dx, label, color in [(1, 'ASD', '#FF6B6B'), (2, 'Control', '#4ECDC4')]:
            ages = valid_ages[valid_ages['DX_GROUP'] == dx]['AGE_AT_SCAN']
            ax.hist(ages, bins=20, alpha=0.6, label=label, color=color)
        ax.set_title('Age Distribution')
        ax.set_xlabel('Age (years)')
        ax.set_ylabel('Frequency')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'Age data not available', ha='center', va='center')
        ax.set_title('Age Distribution')
    
    # 3. Sex Distribution
    ax = axes[0, 2]
    if 'SEX' in manifest_df.columns:
        sex_data = manifest_df.groupby(['DX_GROUP', 'SEX']).size().unstack(fill_value=0)
        sex_data.index = ['ASD', 'Control']
        sex_data.columns = ['Male', 'Female']
        sex_data.plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c'])
        ax.set_title('Sex Distribution by Diagnosis')
        ax.set_xlabel('Diagnosis')
        ax.set_ylabel('Count')
        ax.legend(title='Sex')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)
    else:
        ax.text(0.5, 0.5, 'Sex data not available', ha='center', va='center')
        ax.set_title('Sex Distribution')
    
    # 4. Site Distribution
    ax = axes[1, 0]
    if 'SITE_ID' in manifest_df.columns:
        site_counts = manifest_df['SITE_ID'].value_counts().head(10)
        site_counts.plot(kind='barh', ax=ax, color='#9b59b6')
        ax.set_title('Top 10 Sites by Subject Count')
        ax.set_xlabel('Number of Subjects')
    else:
        ax.text(0.5, 0.5, 'Site data not available', ha='center', va='center')
        ax.set_title('Site Distribution')
    
    # 5. Split Distribution
    ax = axes[1, 1]
    split_dx = manifest_df.groupby(['split', 'DX_GROUP']).size().unstack(fill_value=0)
    split_dx.columns = ['ASD', 'Control']
    split_dx.plot(kind='bar', ax=ax, color=colors)
    ax.set_title('Train/Val/Test Split Distribution')
    ax.set_xlabel('Split')
    ax.set_ylabel('Number of Subjects')
    ax.legend(title='Diagnosis')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)
    
    # 6. Class Balance per Split
    ax = axes[1, 2]
    split_ratios = []
    for split in ['train', 'val', 'test']:
        split_data = manifest_df[manifest_df['split'] == split]
        asd = len(split_data[split_data['DX_GROUP'] == 1])
        total = len(split_data)
        ratio = asd / total if total > 0 else 0
        split_ratios.append(ratio)
    
    ax.bar(['Train', 'Val', 'Test'], split_ratios, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax.axhline(0.5, color='black', linestyle='--', label='Perfect Balance')
    ax.set_title('ASD Ratio per Split')
    ax.set_ylabel('Proportion of ASD')
    ax.set_ylim([0, 1])
    ax.legend()
    
    plt.tight_layout()
    return fig


def analyze_image_data():
    """Analyze processed image data."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Image Data Analysis', fontsize=16, fontweight='bold')
    
    # Check different locations for images
    image_locations = [
        DATA_IMAGES,
        DATA_FINAL / 'train' / 'images',
        DATA_FINAL / 'val' / 'images',
        DATA_FINAL / 'test' / 'images'
    ]
    
    all_images = []
    for loc in image_locations:
        if loc.exists():
            all_images.extend([f.name for f in loc.glob("*.png")])
    
    if not all_images:
        for ax in axes:
            ax.text(0.5, 0.5, 'No image data found', ha='center', va='center')
        return fig
    
    # Extract subject IDs and slice info
    subject_slices = Counter()
    z_positions = []
    
    for img_name in all_images:
        try:
            sub_id, z_str = img_name.rsplit('_z', 1)
            z_idx = int(z_str.split('.')[0])
            subject_slices[sub_id] += 1
            z_positions.append(z_idx)
        except:
            continue
    
    # Plot 1: Slices per Subject
    ax = axes[0]
    slice_distribution = Counter(subject_slices.values())
    ax.bar(slice_distribution.keys(), slice_distribution.values(), color='#3498db')
    ax.set_title('Slices per Subject Distribution')
    ax.set_xlabel('Number of Slices')
    ax.set_ylabel('Number of Subjects')
    ax.axvline(5, color='red', linestyle='--', label='Target: 5 slices')
    ax.legend()
    
    # Plot 2: Z-Position Distribution
    ax = axes[1]
    ax.hist(z_positions, bins=50, color='#2ecc71', edgecolor='black')
    ax.set_title('Z-Slice Position Distribution')
    ax.set_xlabel('Z Index')
    ax.set_ylabel('Frequency')
    
    # Plot 3: Completeness Summary
    ax = axes[2]
    complete = sum(1 for count in subject_slices.values() if count == 5)
    incomplete = sum(1 for count in subject_slices.values() if count < 5)
    over = sum(1 for count in subject_slices.values() if count > 5)
    
    ax.bar(['Complete\n(5 slices)', 'Incomplete\n(< 5)', 'Extra\n(> 5)'], 
           [complete, incomplete, over], 
           color=['#2ecc71', '#e74c3c', '#f39c12'])
    ax.set_title('Subject Completeness')
    ax.set_ylabel('Number of Subjects')
    
    for i, count in enumerate([complete, incomplete, over]):
        ax.text(i, count, str(count), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    return fig


def analyze_features(temporal_features=None, spatial_features=None):
    """Analyze extracted features."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Feature Analysis', fontsize=16, fontweight='bold')
    
    # Temporal Features
    if temporal_features is not None and temporal_features.exists():
        df_temp = pd.read_csv(temporal_features)
        feature_cols = [c for c in df_temp.columns if c != 'subject_id']
        
        # Plot 1: Feature Distributions
        ax = axes[0, 0]
        sample_features = df_temp[feature_cols].iloc[:, :20].values.flatten()
        ax.hist(sample_features, bins=50, color='#9b59b6', edgecolor='black')
        ax.set_title('Sample Feature Value Distribution')
        ax.set_xlabel('Feature Value')
        ax.set_ylabel('Frequency')
        
        # Plot 2: Feature Correlations
        ax = axes[0, 1]
        sample_cols = feature_cols[:30]  # Sample for visualization
        corr_matrix = df_temp[sample_cols].corr()
        sns.heatmap(corr_matrix, ax=ax, cmap='coolwarm', center=0, 
                    square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
        ax.set_title('Feature Correlation Matrix (Sample)')
    else:
        axes[0, 0].text(0.5, 0.5, 'Temporal features not available', 
                        ha='center', va='center')
        axes[0, 1].text(0.5, 0.5, 'Temporal features not available', 
                        ha='center', va='center')
    
    # Spatial Features
    if spatial_features is not None and spatial_features.exists():
        df_spat = pd.read_csv(spatial_features)
        
        # Plot 3: Detection Confidence
        ax = axes[1, 0]
        conf_cols = [c for c in df_spat.columns if '_conf' in c]
        if conf_cols:
            conf_data = df_spat[conf_cols].values.flatten()
            conf_data = conf_data[~np.isnan(conf_data)]
            ax.hist(conf_data, bins=30, color='#e74c3c', edgecolor='black')
            ax.set_title('ROI Detection Confidence Distribution')
            ax.set_xlabel('Confidence Score')
            ax.set_ylabel('Frequency')
            ax.axvline(0.35, color='black', linestyle='--', label='Threshold')
            ax.legend()
        
        # Plot 4: Node Completeness
        ax = axes[1, 1]
        if 'node_count' in df_spat.columns:
            node_counts = df_spat['node_count'].value_counts().sort_index()
            ax.bar(node_counts.index, node_counts.values, color='#3498db')
            ax.set_title('Detected Nodes per Subject')
            ax.set_xlabel('Number of Nodes')
            ax.set_ylabel('Number of Subjects')
            ax.axvline(5, color='red', linestyle='--', label='Target: 5 lobes')
            ax.legend()
    else:
        axes[1, 0].text(0.5, 0.5, 'Spatial features not available', 
                        ha='center', va='center')
        axes[1, 1].text(0.5, 0.5, 'Spatial features not available', 
                        ha='center', va='center')
    
    plt.tight_layout()
    return fig


def analyze_pipeline_completion(manifest_df):
    """Check completion status across pipeline stages."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    stages = []
    counts = []
    colors_list = []
    
    # Stage 1: Manifest entries
    stages.append('Manifest\nEntries')
    counts.append(len(manifest_df))
    colors_list.append('#3498db')
    
    # Stage 2: Time Series
    ts_count = 0
    for _, row in manifest_df.iterrows():
        ts_path = DATA_FINAL / row['split'] / 'time_series' / f"{row['subject_id']}_ts.npy"
        if ts_path.exists():
            ts_count += 1
    stages.append('Time Series\nExtracted')
    counts.append(ts_count)
    colors_list.append('#2ecc71')
    
    # Stage 3: Temporal Features
    if NODE_ATTRIBUTES_TEMPORAL.exists():
        df_temp = pd.read_csv(NODE_ATTRIBUTES_TEMPORAL)
        stages.append('Temporal\nFeatures')
        counts.append(len(df_temp))
        colors_list.append('#9b59b6')
    
    # Stage 4: Spatial Features
    if NODE_FEATURES_3D.exists():
        df_spat = pd.read_csv(NODE_FEATURES_3D)
        stages.append('Spatial\nFeatures')
        counts.append(len(df_spat))
        colors_list.append('#e74c3c')
    
    # Stage 5: Causal Graphs
    if CAUSAL_GRAPHS_DIR.exists():
        graph_count = len(list(CAUSAL_GRAPHS_DIR.glob("*_graph.pt")))
        stages.append('Causal\nGraphs')
        counts.append(graph_count)
        colors_list.append('#f39c12')
    
    ax.bar(stages, counts, color=colors_list)
    ax.set_title('Pipeline Completion Status', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Subjects')
    
    for i, count in enumerate(counts):
        ax.text(i, count, str(count), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    return fig


def main():
    """Run comprehensive EDA."""
    print("="*60)
    print("NEURO-CXG ENHANCED EXPLORATORY DATA ANALYSIS")
    print("="*60 + "\n")
    
    # Check data availability
    available = check_data_availability()
    
    # Load manifest if available
    if not available['Master Manifest']:
        print("❌ Master manifest not found. Run manifest.py first!")
        return
    
    manifest_df = pd.read_csv(MASTER_MANIFEST)
    print(f"Loaded manifest with {len(manifest_df)} subjects\n")
    
    # Generate reports
    figures = []
    
    print("Generating demographic analysis...")
    fig1 = analyze_demographics(manifest_df)
    figures.append(('demographics', fig1))
    
    print("Generating image data analysis...")
    fig2 = analyze_image_data()
    figures.append(('images', fig2))
    
    print("Generating feature analysis...")
    fig3 = analyze_features(NODE_ATTRIBUTES_TEMPORAL, NODE_FEATURES_3D)
    figures.append(('features', fig3))
    
    print("Generating pipeline completion analysis...")
    fig4 = analyze_pipeline_completion(manifest_df)
    figures.append(('pipeline', fig4))
    
    # Save all figures
    output_dir = Path("./eda_outputs")
    output_dir.mkdir(exist_ok=True)
    
    for name, fig in figures:
        output_path = output_dir / f"eda_{name}.png"
        fig.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"✓ Saved {output_path}")
    
    print(f"\n✅ EDA complete! All visualizations saved to {output_dir}/")
    print("="*60)
    
    # Keep plots open for interactive viewing
    plt.show()


if __name__ == "__main__":
    main()

NEURO-CXG ENHANCED EXPLORATORY DATA ANALYSIS

DATA AVAILABILITY CHECK
✗ Master Manifest: /home/nidszxh/Projects/data/metadata/master_manifest.csv
✗ Raw Images: /home/nidszxh/Projects/data/images
✗ Temporal Features: /home/nidszxh/Projects/data/metadata/node_attributes_temporal.csv
✗ 3D Features: /home/nidszxh/Projects/data/metadata/node_features_3d.csv
✗ Causal Graphs: /home/nidszxh/Projects/data/processed/causal_graphs

❌ Master manifest not found. Run manifest.py first!
